In [1]:
import torch
from sorl.gat_sim import GAT, GATConfig, BOS_TOKEN_ID
torch.set_float32_matmul_precision('high')

gat_config = GATConfig.gpt_size("small", [BOS_TOKEN_ID+1, 128])
model = GAT(gat_config)

# ---- load ckpt ----
def local_load_ckpt(model, ckpt_path):
    """load compiled ckpt locally"""
    model_ckpt = torch.load(ckpt_path, map_location="cpu")['model']
    clean_state_dict = {k.replace("_orig_mod.", ""): v for k, v in model_ckpt.items()}
    model.load_state_dict(clean_state_dict)
    return model

ckpt_path = "ckpt/ts-k4-v128.pt"
model = local_load_ckpt(model, ckpt_path)
K = 4


from src.model import GPT_base, GPTConfig

gpt_config = GPTConfig.gpt_size("small", BOS_TOKEN_ID + 1)
base_model = GPT_base(gpt_config)
base_ckpt_path = "ckpt/gpt2-small-ts.pt"
base_model = local_load_ckpt(base_model, base_ckpt_path)

# model = model.to("cuda")
# model = torch.compile(model)

In [2]:
from data.tinystory_local import TinyStoriesDataLoader, collect_rollout_statistics, AbstractionStatistics
from data.tinystory_local import visualize_dynamics
import tiktoken 

# TinyStories Dataset + GPT2 tokenizer
# -------------------------------------
num_stories = 1000
max_len = 1024
doc_len = max_len  + (max_len - 1) // K # <-- doc len contains abstract tokens
loader = TinyStoriesDataLoader(num_stories=num_stories, max_len=max_len, chunk_size=K, device='cpu', split="validation")

batch_size = 8
memory_span = 1792
attn_blocksize = 1792
max_iterations = 2

# ---- stat collection ---
abs_stats = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

# --- tokenizer --- 
enc = tiktoken.get_encoding("gpt2")
eot = enc._special_tokens['<|endoftext|>']

Loading 1000 stories from TinyStories validation...
Loaded 1000 stories, 193538 tokens total, 0.74 MB
Collected 39365 unique 4-chunks


In [ ]:
# Idea #0. Get it to talk shit
# Idea #1. Check for 'search advantage' scaling
# Idea #2. Design proper Catastrophic forgetting probing experiment

# loader.stories

In [3]:
# (I). Generate with SoRL trained on TinyStories Dataset
# -------------------------------------------------
from sorl.neo_utils import generate
from data.tinystory_local import visualize_interleaved_alignment    

min_temperature = 0.0
tokens, doc_ids = loader.get_batch(1)

# idx = tokens[:, :15].clone()
idx = torch.tensor(enc.encode("Chrismas is coming soon, ")).unsqueeze(0)

img_frames = []
for i in range(120): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, temperature=torch.tensor(min_temperature))
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]
    img = visualize_interleaved_alignment(idx, model, enc, K=K, max_chunks=8)
    img_frames.append(img)

# ---- save to gif ----
if len(img_frames) > 0:
    img_frames[0].save(
        'tiny-tiny-stories-generation.gif',
        save_all=True,
        append_images=img_frames[1:],
        duration=400,  # milliseconds sper frame
        loop=0  # 0 = infinite loop
    )
    print(f"Saved GIF with {len(img_frames)} frames")

Saved GIF with 120 frames


In [ ]:
from huggingface_hub import HfApi

# ---- upload ckpt to huggingface ----

api = HfApi()
# api.create_repo(repo_id="Ksgk-fy/sorl", repo_type="model", private=False)

# api.upload_file(
#     path_or_fileobj="ckpt/ts-k4-v128.pt",
#     path_in_repo="ts-k4-v128.pt",
#     repo_id="Ksgk-fy/sorl",
#     repo_type="model"
# )

# api.upload_file(
#     path_or_fileobj="ckpt/gpt2-small-ts.pt",
#     path_in_repo="gpt2-small-ts.pt",
#     repo_id="Ksgk-fy/sorl",
#     repo_type="model"
# )

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Ksgk-fy/sorl/commit/8347005f4ca8e424acb10747c15863e00eb1aaf2', commit_message='Upload gpt2-small-ts.pt with huggingface_hub', commit_description='', oid='8347005f4ca8e424acb10747c15863e00eb1aaf2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Ksgk-fy/sorl', endpoint='https://huggingface.co', repo_type='model', repo_id='Ksgk-fy/sorl'), pr_revision=None, pr_num=None)

In [3]:
from sorl.forget import compute_abs_stats_v3

# (II). Search Advantage Scaling checks || Validation set

n = 5
temperature = torch.tensor([0.0] + [5.0] * (n - 1))

# ---- statistics on info gain reward ---
i = 0
batch_size = 2

from tqdm import tqdm

# initialize lists to collect stats if not already defined
base_traj_loss = []
greedy_traj_loss = []
search_traj_loss = []
ig_greedy = []
ig_search = []
adv_greedy = []
adv_search = []
abs_adv_greedy = []
abs_adv_search = []


with tqdm(total=len(loader.stories), desc="Processing batches") as pbar:
    while i < len(loader.stories): 
        batch_indices = torch.arange(i, min(i + batch_size, len(loader.stories)))
        loader.get_specific(batch_indices) 
        i += batch_size
        tokens, doc_ids = loader.get_specific(batch_indices) 
        # base_loss, greedy_loss, search_loss, greedy_adv, greedy_abs_adv, search_adv, search_abs_adv, greedy_info_gain, search_info_gain = compute_abs_stats_v2(
        #     tokens, model, n=n, K=K, max_iterations=max_iterations, memory_span=memory_span, 
        #     attn_blocksize=attn_blocksize, temperature=temperature, truncate_seq_len=False
        # )

        base_loss, greedy_loss, search_loss, greedy_adv, greedy_abs_adv, search_adv, search_abs_adv, greedy_info_gain, search_info_gain = compute_abs_stats_v3(
            tokens, model, base_model, n=n, K=K, max_iterations=max_iterations, memory_span=memory_span, 
            attn_blocksize=attn_blocksize, temperature=temperature, truncate_seq_len=False
        )

        ig_greedy.append(greedy_info_gain.item())
        ig_search.append(search_info_gain.item())
        adv_greedy.append(greedy_adv.item())
        adv_search.append(search_adv.item())
        abs_adv_greedy.append(greedy_abs_adv.item())
        abs_adv_search.append(search_abs_adv.item())
        base_traj_loss.append(base_loss.item())
        greedy_traj_loss.append(greedy_loss.item())
        search_traj_loss.append(search_loss.item())
        pbar.update(len(batch_indices))
    

print("base traj loss:", torch.tensor(base_traj_loss).mean().item())
print("cond traj loss (greedy):", torch.tensor(greedy_traj_loss).mean().item())
print("cond traj loss (search):", torch.tensor(search_traj_loss).mean().item())
print("greedy relative adv:", torch.tensor(adv_greedy).mean().item())
print("search relative adv:", torch.tensor(adv_search).mean().item())
print("greedy absolute adv:", torch.tensor(abs_adv_greedy).mean().item())
print("search absolute adv:", torch.tensor(abs_adv_search).mean().item())
print("greedy info gain:", torch.tensor(ig_greedy).mean().item())
print("search info gain:", torch.tensor(ig_search).mean().item())

Processing batches:   1%|▏         | 14/1000 [03:31<4:08:48, 15.14s/it]


KeyboardInterrupt: 

In [4]:
(3.84 -  1.73) / 3.84

0.5494791666666666

In [5]:
1.8988 / 3.7048

0.5125242928093284

In [9]:
# 

# Remark #1. 
# - in 'compute_abs_stats', info gain is computed per doc not per-token
# - in 'loss_fn', info gain is computed between 'average per-token' base_traj and 'average per-token' cond_traj in a 'misaligned way' (due to suffix truncation)
# - during training, we got (a). prefix truncation for both base traj & cond traj (b). suffix truncation for cond traj (so that len(cond traj) = len(base traj))
# - these two factors confound with the metric, and lead to different with above metric

# Remark #2. 
# - did some fixes on the 'compute_abs_stats' function, it's easy to mess up 'search' & 'greedy' with 
#   'random rollouts', especially, search is used to represent random rollout aswell as best rolout
#   obtained via search


tensor([1.1009, 1.5971])

In [17]:
compute_abs_stats_v2(tokens, model, n=n, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperature,
                                                                                            truncate_seq_len=False)

(tensor(1.5995),
 tensor([0.0762, 0.0214], grad_fn=<DivBackward0>),
 tensor(0.0432, grad_fn=<DivBackward0>),
 tensor(0.1001),
 tensor(0., grad_fn=<DivBackward0>))

In [ ]:
from sorl.forget import * 
truncate_seq_len = False



In [ ]:
from sorl.forget import collect_forget_data, plot_normalized_correlation_lines, train_forget_vec
from tqdm import tqdm as tqdm

# ---- Memory Interference Experiment ---

abs_stats = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

abs_stats_post = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

# ---- Compute 'Forget Matrix' --- 
forget_mat = torch.zeros(num_stories, num_stories, device=model.device)
num_steps = 40
for train_idx in tqdm(range(num_stories)): 
    forget_vec = train_forget_vec(train_idx, loader, model, abs_stats, abs_stats_post, optimizer, num_steps,
                                max_iterations, memory_span, attn_blocksize, temperature, K, r_min, reward_mode, loss_fn, alpha_abs, alpha_soft_zipf, alpha_topo,
                                ckpt_path="sorl_tinystories.pt")
    forget_mat[train_idx] = forget_vec 

# torch.save(forget_mat, "forget_mat.pt") # save it just in case

# # ---- Visualize 'Forget Matrix' --- 
# correlation_data, ham_corrs = collect_forget_data(forget_mat, abs_stats)

# plot_normalized_correlation_lines(
#     correlation_data, 
#     ham_corrs,
#     xlabel="Abstraction Edit Distance", 
#     ylabel="Forgetting (Δ Perplexity)",
#     title="Distant Abstractions → Less Forgetting (TinyStories)"
# )

In [ ]:
# Exp #1. 
# 'self-organization' of abstraction representation --> will SoRL tries to pull mixing memory apart? 
# Exp #2. 
# setting up tiny memory dataset training pipeline 
# Question #1. 
# What if we use all-rollout SoRL with utility advantage + KL reg terms? 


In [ ]:
from data.tinystory_local import *
from sklearn.decomposition import PCA

cs_sim = abs_stats.compute_cross_doc_logit_sim()
# cs_sim = abs_stats.compute_cross_doc_hamming()
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(cs_sim.cpu().numpy())

# Now use in 3D visualization
train_idx = 0
perplexity = forget_mat[train_idx]
img = visualize_forget_terrain(coords_2d, perplexity, trained_idx=train_idx, step=step)
# img = visualize_perplexity_terrain(coords_2d, perplexity, trained_idx=train_idx, step=step)

In [4]:
# Hypothesis #2. 
# ------------------------------------------------------------
# forget(i | j) is proportional to 1 / d(a_i, a_j)
# dis-similar concept is less likely to be overwritten by each other
# similar concept is more likely to be overwritten by each other
# the emerged abstraction system from SoRL can describe such similarity via d(a_i, a_j)
# ------------------------------------------------------------

# Experiment 
# (a). Train till emergence
# (b). Train with specific order. 
# (c). Record 'forgetting matrix'

import numpy as np
np.array(record['topo_loss'][:5]).mean(), np.array(record['topo_loss'][-5:]).mean()

(-0.7774280548095703, -0.7865824460983276)

In [4]:
if len(img_frames) > 0:
    img_frames[0].save(
        'tinystories_dynamics (select-best per abs SoRL + 1.0 topo reg + 1.0 bigram zipf reg + utility reward scaling.gif',
        save_all=True,
        append_images=img_frames[1:],
        duration=400,  # milliseconds per frame
        loop=0  # 0 = infinite loop
    )
    print(f"Saved GIF with {len(img_frames)} frames")

Saved GIF with 200 frames


In [ ]:
# Generate with SoRL trained on TinyStories Dataset
# -------------------------------------------------
from sorl.neo_utils import generate
from data.tinystory_local import visualize_interleaved_alignment    

min_temperature = 0.0
tokens, doc_ids = loader.get_batch(1)

idx = tokens[:, :15].clone()

img_frames = []
for i in range(30): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, temperature=torch.tensor(min_temperature))
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]
    img = visualize_interleaved_alignment(idx, model, enc, K=K, max_chunks=8)
    img_frames.append(img)

# ---- save to gif ----
if len(img_frames) > 0:
    img_frames[0].save(
        'tiny-tiny-stories-generation.gif',
        save_all=True,
        append_images=img_frames[1:],
        duration=400,  # milliseconds per frame
        loop=0  # 0 = infinite loop
    )
    print(f"Saved GIF with {len(img_frames)} frames")

In [ ]:
# Request #1. 
# -> Full scale experiment on TinyStories & FineWeb

In [ ]:
# marg cond w = 1.0 
# traj_loss: 0.85 | abs_loss: 0.04 | search adv: 21.01% | vocab util: 93.75%  | marg_ent: 2.67 | cond_ent: 0.05 | avg cos sim: 0.44

# soft bigram zipf kl w = 1.0 
# traj_loss: 0.89 | abs_loss: 0.53 | search adv: 25.84% | vocab util: 68.75%  | kl_soft_zipf: 0.24 | avg cos sim: 0.47
# -> visually I observe much less 'repetitions'

# soft bigram zipf kl w = 1.0 & utility reward scaling r = max(p(s|a)/p(s), 1.0)
# traj_loss: 0.79 | abs_loss: 0.40 | search adv: 30.13% | vocab util: 62.50%  | kl_soft_zipf: 0.17 | avg cos sim: 0.44 | topo sim: 0.77 

# soft bigram zip w=1.0 & utility reward scaling & topo reg w=1.0
# traj loss: 1.50 | search adv: 20% | avg cos sim: 0.24 | topo sim: 0.53
# => degrades utility, no go

